# TD1 — From the Unstructured Web to Structured Entities
**Domain : Science-Fiction (authors & works)**

Ce notebook importe les fonctions depuis `src/crawl/crawler.py` et `src/ie/ner.py`.

In [7]:
import warnings
warnings.filterwarnings("ignore")

import os
os.environ["PYTHONWARNINGS"] = "ignore"
import numpy as np
np.warnings = None
import sys
sys.path.append("../")  # pour accéder au dossier src/

from src.crawl.crawler import crawl_and_clean, load_crawled_data, SEED_URLS
from src.ie.ner import load_nlp_model, extract_entities, extract_relations

## Phase 1 — Crawling & Cleaning

In [8]:
# Crawl les seed URLs Sci-Fi
crawled_data = crawl_and_clean(SEED_URLS, output_file="crawler_output.jsonl")

print(f"\nPages collectées : {len(crawled_data)}")
print(f"Total mots       : {sum(d['word_count'] for d in crawled_data):,}")

✓ Isaac_Asimov                             22202 words
✓ Frank_Herbert                            6363 words
✓ Arthur_C._Clarke                         13161 words
✓ Philip_K._Dick                           13300 words
✓ Ursula_K._Le_Guin                        14495 words
✓ Science_fiction                          13858 words
✓ Dune_(novel)                             16444 words
✓ Victor_Hugo                              26592 words

Saved 8/8 pages to crawler_output.jsonl

Pages collectées : 8
Total mots       : 126,415


## Phase 2 — NER & Relation Extraction

In [9]:
# Charger le modèle spaCy
nlp = load_nlp_model("en_core_web_trf")

spaCy model loaded: core_web_trf


In [10]:
print(f"Pages dans crawled_data : {len(crawled_data)}")
if crawled_data:
    print(f"Exemple texte (100 chars) : {crawled_data[0]['text'][:100]}")

Pages dans crawled_data : 8
Exemple texte (100 chars) : Isaac Asimov
Isaac Asimov (/ˈæzɪmɒv/ AZ-im-ov;[b][c] c. January 2, 1920[a] – April 6, 1992) was an A


In [11]:
# Extraire les entités
df_entities = extract_entities(crawled_data, nlp, output_csv="extracted_knowledge_scifi.csv")

print("\nDistribution par type :")
print(df_entities["type"].value_counts())

✓ Isaac_Asimov                             86 entities
✓ Frank_Herbert                            133 entities
✓ Arthur_C._Clarke                         89 entities
✓ Philip_K._Dick                           118 entities
✓ Ursula_K._Le_Guin                        89 entities
✓ Science_fiction                          30 entities
✓ Dune_(novel)                             80 entities
✓ Victor_Hugo                              56 entities

Total unique entities: 530 → saved to extracted_knowledge_scifi.csv

Distribution par type :
type
DATE           166
PERSON         149
WORK_OF_ART     96
ORG             64
GPE             55
Name: count, dtype: int64


In [12]:
# Extraire les relations (triples sujet→verbe→objet)
df_relations = extract_relations(crawled_data, nlp, output_csv="extracted_relations_scifi.csv")

print("\nSample triples :")
print(df_relations[["subject", "relation", "object"]].head(10).to_string(index=False))

Extracted 174 unique triples → saved to extracted_relations_scifi.csv

Sample triples :
 subject   relation    object
  Asimov      write mysteries
   books        win     Award
      It transplant   History
   which    include     works
      he       link    future
      He      write   stories
  Asimov      write    series
Examples    include     Guide
  father      spell        it
    This    inspire       one


## Ambiguity Analysis & Scaling Reflection

Voir le rapport final (section 1.4 et 1.5) pour les 3 exemples d'ambiguïté et la réflexion sur le passage à 10 000 pages.